# Full Training — Detector + Classifier

กด `Runtime → Run all`. Detector (~60 min) แล้วต่อ Classifier (~30 min).
Models เซฟลง Drive: `data check lot/<run>.pt` + `data classify check lot/models/classifier.pt`.

## ── Detector (YOLO) ──

In [ ]:
# 1. ติดตั้ง YOLO
!pip install ultralytics -q

# 2. เชื่อมต่อ Google Drive
# Clear the mount point if it contains files from a previous attempt
!rm -rf /content/drive/*
from google.colab import drive
drive.mount('/content/drive')

print("ติดตั้งและเชื่อมต่อ Google Drive สำเร็จ")

In [ ]:
#v8-12
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

print("เริ่มเทรนโมเดล")
results = model.train(
    data='/content/drive/MyDrive/data check lot/data.yaml',
    epochs=250,
    imgsz=1024,
    batch=16,
    name='ai_check lot v3',
    patience=30,

    # การปรับแต่งรูปทรง
    degrees=180.0,  # หมุนได้ครบ 360° (-180 ถึง +180)
    translate=0.1,
    scale=0.2,
    mosaic=0.5,
    shear=2.0,
)
print("🎉 เทรนเสร็จสมบูรณ์!")

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os
import numpy as np

# 1. โหลดโมเดล
best_model = YOLO('/content/runs/detect/ai_check lot v3/weights/best.pt')
print("ทดสอบ model")

# 2. วัดผล
metrics = best_model.val()

# 3. mAP รวม
map50 = metrics.box.map50
print("\n" + "="*50)
print(f"🎯 ความแม่นยำรวม (mAP50): {map50 * 100:.2f}%")
print("="*50)

# 4. ดึงค่า Precision / Recall / Accuracy ต่อ class
print("\n📊 ผลลัพธ์รายคลาส:")
print("-"*70)
print(f"{'Class':<25} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'F1-Score':>10}")
print("-"*70)

class_names = best_model.names  # dict {0: 'classA', 1: 'classB', ...}

# metrics.box.p = precision ต่อ class, metrics.box.r = recall ต่อ class
precision_per_class = metrics.box.p      # array shape: (num_classes,)
recall_per_class    = metrics.box.r      # array shape: (num_classes,)
ap50_per_class      = metrics.box.ap50   # array shape: (num_classes,)

for i, name in class_names.items():
    p  = precision_per_class[i]
    r  = recall_per_class[i]
    ap = ap50_per_class[i]
    f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
    print(f"{name:<25} {p*100:>9.2f}% {r*100:>9.2f}% {ap*100:>9.2f}% {f1*100:>9.2f}%")

print("-"*70)

# 5. ค่าเฉลี่ยรวมทุก class (Macro Average)
mean_p  = np.mean(precision_per_class)
mean_r  = np.mean(recall_per_class)
mean_ap = np.mean(ap50_per_class)
mean_f1 = (2 * mean_p * mean_r / (mean_p + mean_r)) if (mean_p + mean_r) > 0 else 0.0
print(f"{'📌 Macro Average':<25} {mean_p*100:>9.2f}% {mean_r*100:>9.2f}% {mean_ap*100:>9.2f}% {mean_f1*100:>9.2f}%")
print("="*70)

# 6. Accuracy จาก Confusion Matrix (TP ทั้งหมด / ทั้งหมด)
# YOLO เก็บ confusion matrix ใน metrics.confusion_matrix.matrix
cm = metrics.confusion_matrix.matrix  # shape: (num_classes+1, num_classes+1)
tp_total  = np.trace(cm)              # TP ทุก class รวมกัน
all_total = cm.sum()
overall_accuracy = tp_total / all_total if all_total > 0 else 0.0
print(f"\n🎯 Overall Accuracy (จาก Confusion Matrix): {overall_accuracy*100:.2f}%")
print("   (TP ทั้งหมด / Prediction ทั้งหมด รวม Background)")

# 7. แสดง Confusion Matrix
save_dir = metrics.save_dir
cm_path  = os.path.join(save_dir, 'confusion_matrix.png')
if os.path.exists(cm_path):
    print("\n📊 Confusion Matrix:")
    display(Image(filename=cm_path, width=800))

In [ ]:
import shutil, json, os
from pathlib import Path

DRIVE_DET = '/content/drive/MyDrive/data check lot'
source_model = '/content/runs/detect/ai_check lot v3/weights/best.pt'
destination_model = f'{DRIVE_DET}/full_detector.pt'
shutil.copy(source_model, destination_model)
print('saved detector to', destination_model)

train_imgs = len(list(Path(f'{DRIVE_DET}/train/images').glob('*')))
val_imgs   = len(list(Path(f'{DRIVE_DET}/val/images').glob('*')))
eval_data = {
    'detector_mAP_50': float(metrics.box.map50),
    'precision':       float(metrics.box.mp),
    'recall':          float(metrics.box.mr),
    'epochs':          int(getattr(results, 'epoch', 0) or 0) or 300,
    'imgsz':           640,
    'train_count':     train_imgs,
    'val_count':       val_imgs,
}
# eval.json written LAST — its presence is the "training done" signal
with open(f'{DRIVE_DET}/eval.json', 'w', encoding='utf-8') as f:
    json.dump(eval_data, f, ensure_ascii=False, indent=2)
print('wrote eval.json:', eval_data)

## ── Classifier (EfficientNet-V2-S) ──

In [ ]:
from pathlib import Path

# ── PATH ──────────────────────────────────────────────────────────────────────
DRIVE_ROOT  = Path('/content/drive/MyDrive/data classify check lot')
IMAGES_DIR  = DRIVE_ROOT / 'images'
MODEL_OUT   = DRIVE_ROOT / 'models' / 'classifier.pt'

# ── HYPERPARAMETERS ───────────────────────────────────────────────────────────
CLASSES      = sorted([d.name for d in IMAGES_DIR.iterdir() if d.is_dir()])
IMG_SIZE     = 384      # V2-S pretrained ที่ 384 — ได้ผลดีกว่า 224 ชัดเจน
EPOCHS       = 50       # Stage1=25 epochs, Stage2=25 epochs
BATCH_SIZE   = 16       # T4 + IMG_SIZE 384
LR           = 3e-3     # head training
LR_FINETUNE  = 5e-5     # fine-tune ช้าลง ลด overfit
VAL_SPLIT    = 0.2
SEED         = 42
PATIENCE     = 10       # V2-S ต้องการ patience มากขึ้น
WEIGHT_DECAY = 1e-3     # เพิ่มจาก 1e-4

# สร้าง models/ ถ้ายังไม่มี
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
print('IMAGES_DIR:', IMAGES_DIR)
print('MODEL_OUT :', MODEL_OUT)

In [ ]:
import logging
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, random_split
from torchvision import models, transforms
from tqdm.notebook import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s — %(message)s')
logger = logging.getLogger(__name__)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
total = 0
for cls in CLASSES:
    d = IMAGES_DIR / cls
    imgs = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp']:
        imgs += list(d.glob(ext))
    total += len(imgs)
    print(f'{cls:20s}: {len(imgs)} images')
print(f'{"TOTAL":20s}: {total} images')

In [ ]:
TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.TrivialAugmentWide(),               # policy-based augmentation — effective สำหรับ small dataset
    # 4-way rotation — รับมือรูปพลิก 90/180/270 องศา (แต่ละทิศมีโอกาส 25%)
    transforms.RandomChoice([
        transforms.Lambda(lambda x: x),           # 0° (ปกติ)
        transforms.RandomRotation((90, 90)),       # 90°
        transforms.RandomRotation((180, 180)),     # 180°
        transforms.RandomRotation((270, 270)),     # 270°
    ]),
    transforms.RandomRotation(15),                 # jitter เล็กน้อยบนทุก orientation
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomAffine(degrees=20, shear=15, scale=(0.85, 1.15)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.4),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3),
    transforms.RandomSolarize(threshold=128, p=0.2),       # จำลองรูปที่ถ่ายแสงจ้า
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),  # จำลองรูปเบลอ/ชัด
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

VAL_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class LotImageDataset(Dataset):
    """โหลดรูปจาก images/<class>/ และ assign label ตาม class index"""

    def __init__(self, images_dir: Path, classes: list, transform=None):
        self.transform = transform
        self.samples = []
        for idx, cls in enumerate(classes):
            cls_dir = images_dir / cls
            if not cls_dir.exists():
                logger.warning('Directory not found: %s', cls_dir)
                continue
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.webp'):
                for img_path in cls_dir.glob(ext):
                    self.samples.append((img_path, idx))
        logger.info('Dataset: %d images across %d classes', len(self.samples), len(classes))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
def build_model(num_classes: int) -> nn.Module:
    """EfficientNet-V2-S pretrained, replace classifier head with Dropout"""
    model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

def freeze_backbone(model: nn.Module) -> None:
    for name, param in model.named_parameters():
        param.requires_grad = 'classifier' in name

def unfreeze_all(model: nn.Module) -> None:
    for param in model.parameters():
        param.requires_grad = True

In [ ]:
def run_epoch(model, loader, criterion, optimizer, phase):
    is_train = phase == 'train'
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for images, labels in tqdm(loader, desc=phase, leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train and optimizer:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += len(labels)
    return total_loss / total, correct / total

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
torch.manual_seed(SEED)

# Dataset
full_dataset = LotImageDataset(IMAGES_DIR, CLASSES, transform=TRAIN_TRANSFORMS)
all_indices  = list(range(len(full_dataset)))
all_labels   = [full_dataset.samples[i][1] for i in all_indices]

# Stratified split — ทุก class ได้สัดส่วน 80/20 เท่ากัน
train_indices, val_indices = train_test_split(
    all_indices,
    test_size=VAL_SPLIT,
    stratify=all_labels,
    random_state=SEED,
)

train_ds = torch.utils.data.Subset(full_dataset, train_indices)

val_full = LotImageDataset(IMAGES_DIR, CLASSES, transform=VAL_TRANSFORMS)
val_ds   = torch.utils.data.Subset(val_full, val_indices)

# แสดงจำนวนรูปต่อ class หลัง split
train_labels = [full_dataset.samples[i][1] for i in train_indices]
val_labels   = [full_dataset.samples[i][1] for i in val_indices]
print('Class split (train / val):')
for idx, cls in enumerate(CLASSES):
    t = train_labels.count(idx)
    v = val_labels.count(idx)
    print(f'  {cls:25s}: {t:3d} train / {v:2d} val')

# WeightedRandomSampler
class_counts = torch.zeros(len(CLASSES))
for lbl in train_labels:
    class_counts[lbl] += 1
class_weights  = 1.0 / class_counts.clamp(min=1)
sample_weights = torch.tensor([class_weights[lbl] for lbl in train_labels])
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,  num_workers=2, pin_memory=True)

class_weights_loss = class_weights.to(DEVICE)
model = build_model(num_classes=len(CLASSES)).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights_loss, label_smoothing=0.1)

best_val_acc = 0.0
patience_counter = 0
STAGE1_EPOCHS = EPOCHS // 2
STAGE2_EPOCHS = EPOCHS - STAGE1_EPOCHS

# ── Stage 1: train head only ──────────────────────────────────────────────────
print(f'\n=== Stage 1: head only ({STAGE1_EPOCHS} epochs) ===')
freeze_backbone(model)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)

for epoch in range(STAGE1_EPOCHS):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, 'train')
    val_loss, val_acc     = run_epoch(model, val_loader,   criterion, None,      'val')
    print(f'Epoch {epoch+1:2d}/{STAGE1_EPOCHS} | train loss={train_loss:.4f} acc={train_acc:.3f} | val loss={val_loss:.4f} acc={val_acc:.3f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({'model_state': model.state_dict(), 'classes': CLASSES}, MODEL_OUT)
        print(f'  -> Saved best model (val_acc={best_val_acc:.3f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  -> Early stopping at epoch {epoch+1}')
            break

# ── Stage 2: fine-tune all layers (AdamW + OneCycleLR) ───────────────────────
print(f'\n=== Stage 2: fine-tune all ({STAGE2_EPOCHS} epochs) ===')
unfreeze_all(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR_FINETUNE * 10,
    steps_per_epoch=len(train_loader),
    epochs=STAGE2_EPOCHS,
    pct_start=0.3,
)
patience_counter = 0

for epoch in range(STAGE2_EPOCHS):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, 'train')
    val_loss, val_acc     = run_epoch(model, val_loader,   criterion, None,      'val')
    scheduler.step()
    print(f'Epoch {epoch+1:2d}/{STAGE2_EPOCHS} | train loss={train_loss:.4f} acc={train_acc:.3f} | val loss={val_loss:.4f} acc={val_acc:.3f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({'model_state': model.state_dict(), 'classes': CLASSES}, MODEL_OUT)
        print(f'  -> Saved best model (val_acc={best_val_acc:.3f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  -> Early stopping at epoch {epoch+1}')
            break

print(f'\nDone! Best val accuracy: {best_val_acc:.3f}')
print(f'Model saved to: {MODEL_OUT}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# โหลด best model
ckpt = torch.load(MODEL_OUT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=CLASSES))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=35, ha='right')
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.colorbar(im)
plt.tight_layout()
plt.savefig(DRIVE_ROOT / 'confusion_matrix.png', dpi=150)
plt.show()
print('Saved confusion_matrix.png to Drive')